    "# 🦕 DINO SDK - IngestionEngine Demonstration
",
    "
",
    "Este notebook demonstra como usar o **IngestionEngine** do DINO SDK v1.2.0 para ingestão de arquivos CSV usando AutoLoader e Unity Catalog.
",
    "
",
    "## ✨ **Características do IngestionEngine**:
",
    "
",
    "- ✅ **AutoLoader**: Para ingestão batch e streaming
",
    "- ✅ **Unity Catalog**: Integração completa
",
    "- ✅ **Todas as colunas STRING**: Para arquivos CSV
",
    "- ✅ **Metadados automáticos**: Rastreamento de origem
",
    "- ✅ **Liquid Clustering**: Otimização automática com `CLUSTER BY AUTO`
",
    "- ✅ **Schema evolution**: Configurável
",
    "
",
    "### 📋 **Baseado nas classes Carlton**:
",
    "- `DataReader` → AutoLoader configuration
",
    "- `DataSaver` → Unity Catalog + Delta Lake
",
    "- Adaptado para DINO SDK v1.2.0
",
    "
",
    "### 🚀 **Liquid Clustering Benefits**:
",
    "- **Otimização automática** baseada em padrões de consulta
",
    "- **Zero manutenção** - Databricks gerencia automaticamente
",
    "- **Performance superior** em consultas sem configuração manual
",
    "- **Evolução dinâmica** conforme padrões de acesso mudam"

## 1. Import Classes and Initialize

Primeiro, vamos importar as classes necessárias do DINO SDK.

In [ ]:
# Importar classes do DINO SDK IngestionEngine
from src.dino_sdk.ingestion_engine import (
    IngestionEngine, 
    IngestionConfig, 
    ingest_csv_to_unity_catalog,
    ConfigValidator
)

print("🦕 DINO SDK - IngestionEngine Demo")
print("=" * 45)
print("📦 Imports realizados com sucesso!")

# Verificar Spark (já disponível no Databricks)
print(f"✅ Spark Session: {spark.version}")
print(f"🔗 Spark Context: {spark.sparkContext.appName}")

## 2. Method 1: Quick Ingestion with Convenience Function

A maneira mais **rápida** de fazer ingestão usando a função de conveniência.

In [ ]:
# Configurações de exemplo para ingestão rápida
SOURCE_PATH = "/mnt/data/csv_files/"  # Ajuste para seu ambiente
CATALOG_NAME = "data_master_dev_dbw"
SCHEMA_NAME = "dino_raw_data"
TABLE_NAME = "customer_data_demo"

print("⚡ Ingestão Rápida com Função de Conveniência")
print("=" * 50)
print(f"📁 Origem: {SOURCE_PATH}")
print(f"🎯 Destino: {CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")

try:
    # Ingestão batch usando função de conveniência
    print("\n📥 Iniciando ingestão batch...")
    
    result = ingest_csv_to_unity_catalog(
        spark=spark,
        source_path=SOURCE_PATH,
        catalog_name=CATALOG_NAME,
        schema_name=SCHEMA_NAME,
        table_name=TABLE_NAME,
        file_header=True,
        file_delimiter=",",
        type_run="batch"
    )
    
    print("✅ Ingestão batch concluída com sucesso!")
    print("💡 Tabela criada com todas as colunas como STRING")
    
except Exception as e:
    print(f"❌ Erro na ingestão: {str(e)}")
    print("💡 Verifique se o path existe e você tem permissões")

## 3. Method 2: Advanced Configuration with IngestionConfig

Configuração **avançada** usando a classe `IngestionConfig` para máximo controle.

In [ ]:
# Criar configuração avançada
advanced_config = IngestionConfig(
    # Configurações de origem
    source_path="/mnt/data/complex_csv/",
    file_extension="csv",
    file_header=True,
    file_delimiter="|",  # Delimitador personalizado
    multiline=True,      # Suporte a CSV multiline
    
    # Destino Unity Catalog
    catalog_name="data_master_dev_dbw",
    schema_name="dino_processed_data", 
    table_name="advanced_customer_data",
    
    # Configurações de execução
    type_run="streaming",  # Streaming ao invés de batch
    trigger_processing_time="30 seconds",
    
    # AutoLoader avançado
    schema_evolution_mode="addNewColumns",
    rescue_data_column="_rescued_data",
    
    # Configurações Spark personalizadas
    custom_spark_config={
        "cloudFiles.maxFilesPerTrigger": "50",
        "cloudFiles.maxBytesPerTrigger": "500m",
        "cloudFiles.validateOptions": "false"
    }
)

print("🔧 Configuração Avançada do IngestionEngine")
print("=" * 50)
print(f"📁 Origem: {advanced_config.source_path}")
print(f"🎯 Destino: {advanced_config.catalog_name}.{advanced_config.schema_name}.{advanced_config.table_name}")
print(f"⚡ Tipo: {advanced_config.type_run}")
print(f"⏱️ Trigger: {advanced_config.trigger_processing_time}")
print(f"🔄 Schema Evolution: {advanced_config.schema_evolution_mode}")
print(f"📊 Delimitador: '{advanced_config.file_delimiter}'")
print(f"📝 Multiline: {advanced_config.multiline}")

# Mostrar configurações customizadas
print("\n⚙️ Configurações Spark customizadas:")
for key, value in advanced_config.custom_spark_config.items():
    print(f"   • {key}: {value}")

## 4. Execute Advanced Ingestion with Full Control

Agora vamos executar a ingestão usando o **IngestionEngine** com configuração completa.

In [ ]:
# Criar instância do IngestionEngine
engine = IngestionEngine(spark)

print("🚀 Executando Ingestão Avançada")
print("=" * 40)

try:
    # Validar configuração antes de executar
    print("🔍 Validando configuração...")
    ConfigValidator.validate_ingestion_config(advanced_config)
    print("✅ Configuração válida!")
    
    # Executar ingestão
    print(f"\n📡 Iniciando ingestão {advanced_config.type_run}...")
    
    streaming_query = engine.ingest(advanced_config)
    
    if streaming_query:
        print(f"✅ Stream iniciado com sucesso!")
        print(f"🆔 Query ID: {streaming_query.id}")
        print(f"📊 Status: {'Ativo' if streaming_query.isActive else 'Inativo'}")
        print(f"🎯 Tabela: {advanced_config.catalog_name}.{advanced_config.schema_name}.{advanced_config.table_name}")
        
        # Mostrar configuração de checkpoint
        print(f"💾 Checkpoint: /tmp/checkpoints/{advanced_config.table_name}")
        
        print("\n💡 Para parar o stream: streaming_query.stop()")
        print("💡 Para monitorar: streaming_query.lastProgress")
    else:
        print("✅ Ingestão batch concluída!")
        
except Exception as e:
    print(f"❌ Erro durante ingestão: {str(e)}")
    print("💡 Verifique logs detalhados para mais informações")

## 4.1. Understanding Liquid Clustering (CLUSTER BY AUTO)

O DINO SDK usa **Liquid Clustering** para otimização automática de performance. Vamos entender como funciona.

In [ ]:
print("⚡ Liquid Clustering - Otimização Automática")
print("=" * 50)

# Demonstrar como verificar o clustering de uma tabela criada
table_name = f"{advanced_config.catalog_name}.{advanced_config.schema_name}.{advanced_config.table_name}"

try:
    print(f"🔍 Analisando clustering da tabela: {table_name}")
    
    # Verificar se a tabela existe primeiro
    tables_df = spark.sql(f"SHOW TABLES IN {advanced_config.catalog_name}.{advanced_config.schema_name}")
    table_exists = any(row.tableName == advanced_config.table_name for row in tables_df.collect())
    
    if table_exists:
        # Obter informações da tabela incluindo clustering
        describe_df = spark.sql(f"DESCRIBE DETAIL {table_name}")
        
        print("📊 Detalhes da tabela com Liquid Clustering:")
        for row in describe_df.collect():
            print(f"   📁 Format: {row.format}")
            print(f"   📂 Location: {row.location}")
            print(f"   🗂️ Provider: {row.provider}")
            if hasattr(row, 'clusteringColumns') and row.clusteringColumns:
                print(f"   ⚡ Clustering: {row.clusteringColumns}")
            else:
                print(f"   ⚡ Clustering: AUTO (Liquid Clustering ativo)")
            break
    else:
        print("⚠️ Tabela ainda não foi criada. Execute a ingestão primeiro.")
        
    # Explicar benefícios do Liquid Clustering
    print("\n🚀 Benefícios do Liquid Clustering:")
    benefits = [
        "✅ Otimização automática baseada em padrões de consulta",
        "✅ Zero manutenção - sem necessidade de gerenciamento manual",
        "✅ Adapta-se automaticamente conforme dados e consultas evoluem", 
        "✅ Melhor performance de leitura sem overhead de configuração",
        "✅ Suporta múltiplas colunas de clustering automaticamente",
        "✅ Compactação automática otimizada"
    ]
    
    for benefit in benefits:
        print(f"   {benefit}")
        
    print("\n💡 SQL gerado pelo DINO SDK:")
    sample_sql = f'''
    CREATE TABLE IF NOT EXISTS {table_name} (
        -- Colunas do arquivo CSV
        column1 STRING,
        column2 STRING,
        -- Metadados DINO
        _rescued STRING,
        dino_ingestion_date DATE,
        dino_ingestion_timestamp TIMESTAMP,
        dino_metadata STRUCT<...>
    )
    USING DELTA
    CLUSTER BY AUTO  ← Liquid Clustering habilitado!
    '''
    print(sample_sql)
        
except Exception as e:
    print(f"❌ Erro ao analisar clustering: {str(e)}")

## 5. Schema Integration with SchemaManager

Vamos integrar com o **SchemaManager** para criar schemas automaticamente antes da ingestão.

In [ ]:
# Importar SchemaManager para integração
from src.dino_sdk.schema_manager import SchemaManager

print("🔗 Integração IngestionEngine + SchemaManager")
print("=" * 50)

# Configurações para pipeline completo
PIPELINE_CATALOG = "data_master_dev_dbw"
PIPELINE_SCHEMA = "dino_pipeline_demo"
PIPELINE_TABLE = "sales_data"

try:
    # Passo 1: Verificar/criar schema
    print("📂 Passo 1: Verificando schema...")
    schema_manager = SchemaManager(PIPELINE_CATALOG, PIPELINE_SCHEMA)
    
    if not schema_manager.schema_exists(spark):
        print(f"🏗️ Criando schema {PIPELINE_CATALOG}.{PIPELINE_SCHEMA}...")
        schema_result = schema_manager.create_schema(spark)
        
        if schema_result['success']:
            print("✅ Schema criado com sucesso!")
        else:
            print(f"❌ Erro ao criar schema: {schema_result['errors']}")
            raise Exception("Schema creation failed")
    else:
        print("✅ Schema já existe")
    
    # Passo 2: Executar ingestão
    print("\n📥 Passo 2: Executando ingestão...")
    
    pipeline_result = ingest_csv_to_unity_catalog(
        spark=spark,
        source_path="/mnt/data/sales_csv/",  # Ajuste para seu ambiente
        catalog_name=PIPELINE_CATALOG,
        schema_name=PIPELINE_SCHEMA,
        table_name=PIPELINE_TABLE,
        file_header=True,
        file_delimiter=",",
        type_run="batch"
    )
    
    print("✅ Pipeline completo executado com sucesso!")
    
    # Passo 3: Validar resultado
    print("\n🔍 Passo 3: Validando resultado...")
    schema_info = schema_manager.get_schema_info(spark)
    
    print(f"📊 Schema existe: {schema_info['schema_exists']}")
    print(f"🗃️ Número de tabelas: {len(schema_info['tables'])}")
    
    if PIPELINE_TABLE in schema_info['tables']:
        print(f"✅ Tabela '{PIPELINE_TABLE}' criada com sucesso!")
        
        # Verificar estrutura da tabela
        table_df = spark.sql(f"DESCRIBE TABLE {PIPELINE_CATALOG}.{PIPELINE_SCHEMA}.{PIPELINE_TABLE}")
        print("\n📋 Estrutura da tabela criada:")
        table_df.show(10, truncate=False)
    else:
        print(f"⚠️ Tabela '{PIPELINE_TABLE}' não encontrada")
        
except Exception as e:
    print(f"❌ Erro no pipeline: {str(e)}")

## 6. Monitoring and Troubleshooting

Ferramentas para **monitoramento** e **troubleshooting** de ingestões.

In [ ]:
print("📊 Ferramentas de Monitoramento")
print("=" * 40)

# Exemplo de configuração para streaming com monitoramento
monitoring_config = IngestionConfig(
    source_path="/mnt/data/streaming_csv/",
    catalog_name="data_master_dev_dbw",
    schema_name="dino_monitoring",
    table_name="streaming_events",
    type_run="streaming",
    trigger_processing_time="10 seconds"
)

try:
    # Iniciar stream para monitoramento
    print("🚀 Iniciando stream para monitoramento...")
    
    monitor_engine = IngestionEngine(spark)
    query = monitor_engine.ingest(monitoring_config)
    
    if query and query.isActive:
        print(f"📡 Stream ativo: {query.id}")
        
        # Simular monitoramento por alguns ciclos
        import time
        
        print("\n📈 Monitorando por 30 segundos...")
        for i in range(3):
            time.sleep(10)
            
            if query.isActive:
                progress = query.lastProgress
                
                if progress:
                    batch_id = progress.get('batchId', 'N/A')
                    input_rows = progress.get('inputRowsPerSecond', 0)
                    processed_rows = progress.get('processedRowsPerSecond', 0)
                    
                    print(f"   📊 Batch {batch_id}: {input_rows:.1f} rows/sec input, {processed_rows:.1f} rows/sec processed")
                else:
                    print(f"   📊 Ciclo {i+1}: Aguardando dados...")
            else:
                print("⚠️ Stream não está mais ativo")
                break
        
        # Informações do stream
        print("\n🔍 Informações detalhadas do stream:")
        print(f"   🆔 ID: {query.id}")
        print(f"   📛 Nome: {query.name or 'Sem nome'}")
        print(f"   ⚡ Status: {'Ativo' if query.isActive else 'Inativo'}")
        print(f"   🎯 Sink: {monitoring_config.catalog_name}.{monitoring_config.schema_name}.{monitoring_config.table_name}")
        
        # Opção para parar o stream
        print("\n💡 Para parar este stream execute: query.stop()")
        
    else:
        print("⚠️ Stream não pôde ser iniciado ou já finalizou")
        
except Exception as e:
    print(f"❌ Erro no monitoramento: {str(e)}")

# Comandos úteis para troubleshooting
print("\n🔧 Comandos Úteis para Troubleshooting:")
print("-" * 45)
troubleshooting_commands = [
    "# Verificar streams ativos",
    "spark.streams.active",
    "",
    "# Parar todos os streams",
    "for s in spark.streams.active: s.stop()",
    "",
    "# Verificar tabela criada",
    "spark.sql('DESCRIBE TABLE catalog.schema.table').show()",
    "",
    "# Contar registros",
    "spark.sql('SELECT COUNT(*) FROM catalog.schema.table').show()",
    "", 
    "# Verificar últimos registros",
    "spark.sql('SELECT * FROM catalog.schema.table ORDER BY dino_ingestion_timestamp DESC LIMIT 10').show()"
]

for cmd in troubleshooting_commands:
    print(cmd)

## 🎯 Resumo do IngestionEngine

### ✅ **Funcionalidades Implementadas**

1. **AutoLoader Integration**: Configuração automática para batch e streaming
2. **Unity Catalog Support**: Criação automática de tabelas no catálogo
3. **Liquid Clustering**: Otimização automática com `CLUSTER BY AUTO`
4. **Schema Evolution**: Configurável (rescue, addNewColumns, etc.)
5. **CSV Optimization**: Todas as colunas mantidas como STRING
6. **Metadata Tracking**: Colunas automáticas de rastreamento (`dino_*`)
7. **Delta Lake**: Com otimização automática de performance
8. **Error Handling**: Validação e tratamento robusto de erros

### 🚀 **Duas Formas de Uso**

**Método 1 - Função de Conveniência** (Rápido):
```python
ingest_csv_to_unity_catalog(
    spark=spark,
    source_path="/path/to/csv/",
    catalog_name="catalog",
    schema_name="schema", 
    table_name="table"
)
```

**Método 2 - Configuração Completa** (Controle total):
```python
config = IngestionConfig(...)
engine = IngestionEngine(spark)
result = engine.ingest(config)
```

### 📊 **Características Técnicas**

- **Baseado em Carlton**: DataReader + DataSaver patterns
- **AutoLoader**: `cloudFiles` format com configurações otimizadas
- **Streaming API**: Mesmo para batch (availableNow=True)
- **Liquid Clustering**: `CLUSTER BY AUTO` para otimização automática
- **Checkpoints**: Automáticos para streaming
- **Rescue Columns**: Para dados malformados

### ⚡ **Liquid Clustering Benefits**

- **Zero Maintenance**: Databricks gerencia automaticamente
- **Performance Otimizada**: Baseada em padrões de consulta reais
- **Evolução Automática**: Adapta-se conforme uso dos dados muda
- **Compactação Inteligente**: Otimização contínua em background

### 🔗 **Integração com SchemaManager**

O IngestionEngine funciona **perfeitamente** com o SchemaManager:
1. SchemaManager cria o schema se necessário
2. IngestionEngine cria a tabela com Liquid Clustering e ingere os dados
3. Pipeline completo automatizado com otimização de performance

**🎉 DINO SDK v1.2.0 - IngestionEngine com Liquid Clustering pronto para produção!**